 # Notebook 06 — LWC Retrieval Uncertainty (Zhu et al. 2019, Eq. 5–8)



 This notebook attaches a **1-sigma uncertainty to every retrieved LWC value**.



 Why this matters: "37 % of my LWC values are negative" is a weak statement,

 because a value of -0.05 g/m3 with an uncertainty of 0.34 g/m3 is perfectly

 consistent with zero. Socuellamos et al. (2024) make exactly this argument:

 they keep their negative values and show that ">95 % of the negative numbers

 can become positive within the uncertainty limits". Once we have sigma_LWC we

 can replace "fraction negative" with "fraction SIGNIFICANTLY negative", which

 is a much more defensible number.



 ## The equations we implement



 Zhu Eq. (6) — random error of one reflectivity measurement, in dB:



     dZ_f = 4.343 / sqrt(dt * PRF_f * N)

            * [ lambda_f * PRF_f / (4*sqrt(pi)*sigma_w_f) + 1/SNR_f^2 + 2/SNR_f ]^(1/2)



 Zhu Eq. (5) — LWC uncertainty of ONE retrieval window:



     dq_L = sqrt(dZ_f1^2 + dZ_f2^2) / [ sqrt(2) * (kappa_f1 - kappa_f2) * dh ]



 Zhu Eq. (7) — reduction from averaging M overlapping windows:



     dq_L_final = dq_L / sqrt(M_eff)



 Zhu Eq. (8) — effective sample size, accounting for window overlap:



     M_eff = M / (1 + 2 * sum_k rho(k))



 ## IMPORTANT: a labelling error in the Zhu paper



 Zhu's text describes `dh` in Eq. (5) as "the range gate spacing (in m)".

 **That is wrong.** Hogan et al. (2005), whose equations Zhu is reproducing,

 uses `(h2 - h1)` — the separation between the two heights spanning the DWR

 gradient — and Hogan Eq. (7) states explicitly:

 *"recognising that h2 - h1 = N*delta (where delta is the range gate spacing)"*.



 Combining Zhu (5)+(6) reproduces Hogan (7) **only** if dh = N * gate_spacing.

 Checked numerically: using the gate spacing instead of the window span

 overestimates the uncertainty by exactly a factor of N.



 So in this notebook `dh` is always the **vertical span of the fitting window**

 = N_gates_in_window * gate_spacing.



 ## A caveat about the theoretical route



 Hogan's Appendix states his Eq. (6) assumes *"the radar is ... of the ordinary

 pulsed type, utilizing a square-law detector and performing linear averages

 with no spectral processing"*. Our radar is an **RPG FMCW** system, which is

 none of those things. The theoretical dZ is therefore best read as a

 **lower bound** on the true noise — what the noise would be if only receiver

 noise mattered. The gap between it and the measured noise tells us how much

 other error (range sidelobes, beam mismatch, clutter) is present.

 ---

 ## Step 0 — Imports and loading

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

sys.path.append(str(Path("../src").resolve()))


In [ ]:
Processed_Dataset = "20230930"

In [ ]:
# The base retrieval inputs, written by notebook 03.
# Contains: Ze_W, Ze_Ka, T_cloud, LWP, DWR, cloud_mask, combined_mask,
#           common_mask, alpha_W, alpha_Ka, kappa_W, kappa_Ka,
#           MeanVel_W, MeanVel_Ka
path_retrieval_input = Path(f"../data/processed/{Processed_Dataset}/retrieval_input_2km_alpha_kappa_calculated.nc")
ds_retrieval = xr.open_dataset(path_retrieval_input)

# The retrieval OUTPUTS, written by the new export cells in notebooks 04 and 05.
# Contains: lwc_guarded, n_windows, dwr_filtered   (from 04)
#           lwc_dq, dq                             (from 05)
path_zhu_output = Path(f"../data/processed/{Processed_Dataset}/retrieval_output_zhu.nc")
path_dq_output = Path(f"../data/processed/{Processed_Dataset}/retrieval_output_dq.nc")

ds_zhu = xr.open_dataset(path_zhu_output)
ds_dq = xr.open_dataset(path_dq_output)

print("Zhu output variables:", list(ds_zhu.data_vars))
print("DQ  output variables:", list(ds_dq.data_vars))


In [ ]:
# Pull out the pieces we will use repeatedly, with explicit names.
# `.values` converts an xarray DataArray into a plain NumPy array. We do this
# where we want fast arithmetic and do not need the coordinate labels.

mask_retrieval = ds_retrieval["common_mask"]          # True = gate used in retrieval
dwr_filtered = ds_zhu["dwr_filtered"]                 # 30 s averaged + despiked DWR
lwc_zhu = ds_zhu["lwc_guarded"]                       # LWC from the Zhu retrieval
n_windows_zhu = ds_zhu["n_windows"]                   # how many windows hit each gate
lwc_dq = ds_dq["lwc_dq"]                              # LWC from the DQ retrieval

kappa_W = ds_retrieval["kappa_W"]
kappa_Ka = ds_retrieval["kappa_Ka"]

# Gate spacing in kilometres.
# `.diff("range")` gives the difference between consecutive range values.
# We take the mean because the spacing can vary slightly between chirps.
range_values_m = ds_retrieval["range"].values.astype(float)
gate_spacing_m = float(np.mean(np.diff(range_values_m)))
gate_spacing_km = gate_spacing_m / 1000.0

print(f"Gate spacing = {gate_spacing_m:.2f} m = {gate_spacing_km:.5f} km")


 ---

 ## Step 1 — Radar parameters



 **Everything in this cell is an assumption.** They are collected in one place

 so they are easy to find, easy to change, and easy to list in the thesis as

 stated assumptions. Replace them with real values as you obtain them.

In [ ]:
SPEED_OF_LIGHT_M_PER_S = 2.99792458e8

frequency_W_hz = 94.0e9
frequency_Ka_hz = 35.0e9

wavelength_W_m = SPEED_OF_LIGHT_M_PER_S / frequency_W_hz     # 3.189 mm
wavelength_Ka_m = SPEED_OF_LIGHT_M_PER_S / frequency_Ka_hz   # 8.565 mm

# --- ASSUMPTION 1: pulse repetition frequency -------------------------------
# Taken from Hogan et al. (2005), who use 6250 Hz at both frequencies in their
# worked example. NOTE: an FMCW radar does not really have a PRF; the
# equivalent quantity is the chirp repetition rate, which differs per chirp.
# This is a placeholder. Ask RPG or read it from the LV1 file header.
prf_W_hz = 6250.0
prf_Ka_hz = 6250.0

# --- ASSUMPTION 2: Doppler spectral width -----------------------------------
# Hogan uses 0.3 m/s for stratocumulus. If your LV1 files contain a spectral
# width variable (often C1SpecWidth etc.), load it and use the real values.
spectral_width_W_m_per_s = 0.3
spectral_width_Ka_m_per_s = 0.3

# --- ASSUMPTION 3: signal-to-noise ratio ------------------------------------
# Given as a LINEAR ratio, not dB.  linear = 10 ** (dB / 10)
# 20 dB is a reasonably good in-cloud value. Section 1b below shows how to
# compute the real SNR per gate from the sensitivity limits instead.
snr_W_linear = 10 ** (20.0 / 10.0)
snr_Ka_linear = 10 ** (20.0 / 10.0)

# --- ASSUMPTION 4: dwell time -----------------------------------------------
# This is the temporal averaging applied before the retrieval.
dwell_time_s = 30.0

# --- Retrieval window -------------------------------------------------------
# N_default in retrieve_lwc_zhu_guarded. This is what sets `dh`.
n_gates_in_window = 9
window_span_km = n_gates_in_window * gate_spacing_km

print(f"lambda_W  = {wavelength_W_m * 1000:.3f} mm")
print(f"lambda_Ka = {wavelength_Ka_m * 1000:.3f} mm")
print(f"Window span dh = {n_gates_in_window} gates "
      f"= {window_span_km * 1000:.0f} m = {window_span_km:.4f} km")
print(f"  (NOT the gate spacing of {gate_spacing_m:.1f} m -- see the note above)")


 ### Step 1b — (optional) real SNR from the sensitivity limits



 If your dataset carries `SLv_W` / `SLv_Ka`, you can compute a real per-gate

 SNR instead of assuming a constant. SNR in dB is simply signal minus noise

 floor when both are in dB:



     SNR_dB = Ze_dBZ - SL_dBZ



 Caveat: this assumes the sensitivity-limit field is calibrated as the

 noise-equivalent reflectivity. That is standard practice but is still

 unconfirmed for this instrument, so treat the result as provisional.

In [ ]:
if "SLv_W" in ds_retrieval and "SLv_Ka" in ds_retrieval:
    # 10*log10 converts the linear mm^6/m^3 sensitivity limit into dBZ
    sensitivity_limit_W_dbz = 10.0 * np.log10(ds_retrieval["SLv_W"])
    sensitivity_limit_Ka_dbz = 10.0 * np.log10(ds_retrieval["SLv_Ka"])

    snr_W_db_field = (ds_retrieval["Ze_W"] - sensitivity_limit_W_dbz).where(mask_retrieval)
    snr_Ka_db_field = (ds_retrieval["Ze_Ka"] - sensitivity_limit_Ka_dbz).where(mask_retrieval)

    # np.nanmedian ignores NaNs. The median is more robust than the mean here
    # because a few very low-SNR gates would drag a mean down a lot.
    snr_W_db_median = float(np.nanmedian(snr_W_db_field.values))
    snr_Ka_db_median = float(np.nanmedian(snr_Ka_db_field.values))

    print(f"Median in-cloud SNR, W-band : {snr_W_db_median:5.1f} dB")
    print(f"Median in-cloud SNR, Ka-band: {snr_Ka_db_median:5.1f} dB")
    print("\nTo use these instead of the assumed 20 dB, uncomment the two lines below.")
    # snr_W_linear  = 10 ** (snr_W_db_median / 10.0)
    # snr_Ka_linear = 10 ** (snr_Ka_db_median / 10.0)
else:
    print("SLv_W / SLv_Ka not in the dataset - keeping the assumed SNR of 20 dB.")
    print("Add them to the notebook 03 export if you want the real values.")


 ---

 ## Step 2 — The EMPIRICAL route: measure the DWR noise from the data



 Before trusting any model, measure the noise that is actually there.



 **Derivation.** Write the measured DWR at gate *i* as

 `D_i = d_i + eps_i`, where `d_i` is the true DWR and `eps_i` is random error

 with variance `sigma^2`. The gate-to-gate difference is



     Delta_i = D_(i+1) - D_i = (d_(i+1) - d_i) + (eps_(i+1) - eps_i)



 **Assumption:** the true DWR barely changes between adjacent gates, so

 `d_(i+1) ≈ d_i`. Then, because the two errors are independent

 (so their covariance is zero):



     Var(Delta) = Var(eps_(i+1)) + Var(eps_i) = 2 * sigma^2



 and therefore `sigma = std(Delta) / sqrt(2)`.

In [ ]:
# Take the DWR only where the retrieval mask is True; everything else is NaN.
dwr_masked_values = dwr_filtered.where(mask_retrieval).values

# Difference between consecutive range gates.
# dwr_masked_values[:, 1:]  -> all times, gates 1..end
# dwr_masked_values[:, :-1] -> all times, gates 0..end-1
# Subtracting them gives DWR(i+1) - DWR(i) for every gate pair.
dwr_gate_differences = dwr_masked_values[:, 1:] - dwr_masked_values[:, :-1]

# np.isfinite is True for real numbers and False for NaN or infinity.
# Using it as an index keeps only the valid entries and flattens the array.
dwr_gate_differences = dwr_gate_differences[np.isfinite(dwr_gate_differences)]

sigma_dwr_measured_db = float(np.std(dwr_gate_differences) / np.sqrt(2.0))

print(f"Valid gate pairs used         : {dwr_gate_differences.size}")
print(f"std of gate-to-gate difference: {np.std(dwr_gate_differences):.3f} dB")
print(f"=> measured per-gate DWR noise: {sigma_dwr_measured_db:.3f} dB")


In [ ]:
# How much of that std is real signal rather than noise?
# A true DWR gradient of ~5 dB/km over one gate contributes:
assumed_true_gradient_db_per_km = 5.0
signal_per_gate_db = assumed_true_gradient_db_per_km * gate_spacing_km
variance_fraction_from_signal = (signal_per_gate_db ** 2) / (np.std(dwr_gate_differences) ** 2)

print(f"A {assumed_true_gradient_db_per_km:.0f} dB/km gradient contributes "
      f"{signal_per_gate_db:.3f} dB per gate,")
print(f"i.e. {variance_fraction_from_signal:.1%} of the measured variance.")
print("Small, so the noise estimate is only mildly inflated by real signal.")


 ---

 ## Step 3 — The THEORETICAL route: Zhu Eq. (6)



 `4.343` is the conversion factor from natural logarithm units (nepers) to

 decibels: `10 / ln(10) = 4.3429`.

In [ ]:
def reflectivity_error_db(wavelength_m, prf_hz, spectral_width_m_per_s,
                          snr_linear, dwell_time_s, n_gates):
    """
    Zhu et al. (2019) Eq. (6) / Hogan et al. (2005) Eq. (6).

    Random error of an averaged reflectivity measurement, in dB.

    Parameters
    ----------
    wavelength_m           : radar wavelength [m]
    prf_hz                 : pulse repetition frequency [Hz]
    spectral_width_m_per_s : Doppler spectral width [m/s]
    snr_linear             : signal-to-noise ratio, LINEAR (not dB)
    dwell_time_s           : temporal averaging applied [s]
    n_gates                : number of range gates grouped together

    Returns
    -------
    Random reflectivity error [dB]
    """
    # Number of independent samples averaged in time.
    n_independent_time_samples = dwell_time_s * prf_hz

    # The three terms inside the square-root bracket of Eq. (6).
    # Term 1 is the finite-bandwidth / spectral-broadening term.
    # Terms 2 and 3 are the SNR-dependent noise terms. They vanish when SNR is
    # large, which is why high SNR matters.
    term_spectral = (wavelength_m * prf_hz
                     / (4.0 * np.sqrt(np.pi) * spectral_width_m_per_s))
    term_snr_squared = 1.0 / snr_linear ** 2
    term_snr_linear = 2.0 / snr_linear

    bracket = term_spectral + term_snr_squared + term_snr_linear

    return (4.343 / np.sqrt(n_independent_time_samples * n_gates)
            * np.sqrt(bracket))


reflectivity_error_W_db = reflectivity_error_db(
    wavelength_W_m, prf_W_hz, spectral_width_W_m_per_s,
    snr_W_linear, dwell_time_s, n_gates_in_window)

reflectivity_error_Ka_db = reflectivity_error_db(
    wavelength_Ka_m, prf_Ka_hz, spectral_width_Ka_m_per_s,
    snr_Ka_linear, dwell_time_s, n_gates_in_window)

# The DWR is a difference of two independent reflectivities, so its error is
# the root-sum-square of the two individual errors.
sigma_dwr_theoretical_db = float(
    np.sqrt(reflectivity_error_W_db ** 2 + reflectivity_error_Ka_db ** 2))

print(f"dZ_W  (Zhu Eq. 6) = {reflectivity_error_W_db:.4f} dB")
print(f"dZ_Ka (Zhu Eq. 6) = {reflectivity_error_Ka_db:.4f} dB")
print(f"=> theoretical DWR noise = {sigma_dwr_theoretical_db:.4f} dB")


 ---

 ## Step 4 — Compare the two routes



 This comparison is itself a result. If the measured noise is far larger than

 the receiver-noise model predicts, then receiver noise is **not** what limits

 the retrieval, and pushing SNR higher will not help much.

In [ ]:
ratio_measured_to_theoretical = sigma_dwr_measured_db / sigma_dwr_theoretical_db

print(f"measured    sigma_DWR = {sigma_dwr_measured_db:.3f} dB")
print(f"theoretical sigma_DWR = {sigma_dwr_theoretical_db:.3f} dB")
print(f"ratio                 = {ratio_measured_to_theoretical:.1f}x")
print()
if ratio_measured_to_theoretical > 3:
    print("The measured noise greatly exceeds the receiver-noise model.")
    print("Receiver noise is NOT the limiting factor. Likely contributors:")
    print("  - range sidelobe leakage (FMCW / pulse compression; Hogan 2005")
    print("    warns this technique needs ~0.03 dB precision and that FMCW")
    print("    radars are unlikely to achieve it)")
    print("  - beam / sampling-volume mismatch between two separate radars")
    print("    (Zhu could neglect this because the SACR2 is a single instrument)")
    print("  - residual clutter or drizzle inside the mask")
    print()
    print("Hogan's stated precision requirement for this technique: ~0.03 dB")
    print(f"Ours: {sigma_dwr_measured_db:.2f} dB "
          f"({sigma_dwr_measured_db / 0.03:.0f}x larger)")


 ---

 ## Step 5 — Zhu Eq. (5): uncertainty of a single retrieval window



     dq_L = sqrt(dZ_f1^2 + dZ_f2^2) / [ sqrt(2) * (kappa_f1 - kappa_f2) * dh ]



 Where does the `sqrt(2)` come from? LWC is derived from a *difference* of two

 DWR values, and the error of a difference of two independent quantities is

 `sqrt(2)` times the individual error. Combined with the factor 2 that

 accounts for two-way attenuation, `sqrt(2)*dDWR / (2*dkappa*dh)` simplifies to

 `dDWR / (sqrt(2)*dkappa*dh)`.

In [ ]:
def lwc_uncertainty_single_window(sigma_dwr_db, delta_kappa, window_span_km):
    """
    Zhu Eq. (5). Uncertainty of one LWC retrieval window.

    sigma_dwr_db   : DWR error [dB]  (= sqrt(dZ_f1^2 + dZ_f2^2))
    delta_kappa    : kappa_W - kappa_Ka  [dB / km / (g/m3)]
    window_span_km : N_gates * gate_spacing, in km   <-- NOT the gate spacing
    """
    return sigma_dwr_db / (np.sqrt(2.0) * delta_kappa * window_span_km)


# delta_kappa is a full 2-D field because kappa depends on temperature.
delta_kappa_field = kappa_W - kappa_Ka

print(f"delta_kappa: min {float(delta_kappa_field.min()):.2f}, "
      f"median {float(delta_kappa_field.median()):.2f}, "
      f"max {float(delta_kappa_field.max()):.2f} dB/km/(g/m3)")

lwc_uncertainty_single_measured = lwc_uncertainty_single_window(
    sigma_dwr_measured_db, delta_kappa_field, window_span_km)

lwc_uncertainty_single_theoretical = lwc_uncertainty_single_window(
    sigma_dwr_theoretical_db, delta_kappa_field, window_span_km)

print(f"\nSingle-window LWC uncertainty (median over the cloud):")
print(f"  from measured noise    : "
      f"{float(lwc_uncertainty_single_measured.where(mask_retrieval).median()):.3f} g/m3")
print(f"  from theoretical noise : "
      f"{float(lwc_uncertainty_single_theoretical.where(mask_retrieval).median()):.3f} g/m3")
print(f"\nZhu reported 0.10-0.65 g/m3 for their clouds.")


 ### Sanity check: what happens with the WRONG dh?



 Demonstrating the factor-N error, so it is documented rather than assumed.

In [ ]:
lwc_uncertainty_wrong_dh = lwc_uncertainty_single_window(
    sigma_dwr_measured_db, delta_kappa_field, gate_spacing_km)

median_correct = float(lwc_uncertainty_single_measured.where(mask_retrieval).median())
median_wrong = float(lwc_uncertainty_wrong_dh.where(mask_retrieval).median())

print(f"dh = window span ({window_span_km*1000:.0f} m) -> {median_correct:.3f} g/m3   [CORRECT]")
print(f"dh = gate spacing ({gate_spacing_m:.0f} m) -> {median_wrong:.3f} g/m3   [WRONG]")
print(f"ratio = {median_wrong / median_correct:.1f}  (equals N = {n_gates_in_window})")


 ---

 ## Step 6 — Zhu Eq. (7) and (8): averaging over overlapping windows



     dq_L_final = dq_L / sqrt(M_eff),   M_eff = M / (1 + 2 * sum_k rho(k))



 `M` is the number of windows contributing to a gate — this is exactly the

 `n_windows` array the guarded retrieval returns.



 The windows overlap, so the M estimates are **not independent**. Two windows

 offset by `k` gates share `(N-k)/N` of their data, so a first-order model of

 the autocorrelation is `rho(k) = 1 - k/N`.



 Working the sum through: `sum_(k=1)^(N-1) (1 - k/N) = (N-1)/2`, so the

 denominator becomes `1 + 2*(N-1)/2 = N`, giving `M_eff = M / N`.



 Since `M <= N` at every gate, `M_eff <= 1` and the correction does essentially

 nothing. **That is the correct answer, not a bug** — fully overlapping windows

 carry almost no independent information. Zhu report the same behaviour:

 *"LWC uncertainty was typically nearly constant with height."*

In [ ]:
def effective_sample_size(n_windows_array, n_gates_in_window):
    """Zhu Eq. (8), with rho(k) = 1 - k/N for overlapping sliding windows."""
    lag_values = np.arange(1, n_gates_in_window)
    autocorrelation_sum = np.sum(1.0 - lag_values / n_gates_in_window)
    denominator = 1.0 + 2.0 * autocorrelation_sum

    # np.maximum(..., 1.0) applies elementwise: the effective sample size can
    # never be less than one measurement.
    return np.maximum(n_windows_array / denominator, 1.0)


n_windows_values = n_windows_zhu.values.astype(float)
m_effective_values = effective_sample_size(n_windows_values, n_gates_in_window)

print(f"windows per gate M   : median {np.nanmedian(n_windows_values[n_windows_values>0]):.1f}, "
      f"max {np.nanmax(n_windows_values):.0f}")
print(f"effective M_eff      : median {np.nanmedian(m_effective_values):.2f}, "
      f"max {np.nanmax(m_effective_values):.2f}")
print(f"resulting reduction  : factor {np.sqrt(np.nanmedian(m_effective_values)):.2f}")
print("\n(As explained above, ~1.0 is expected for step-1 sliding windows.)")


In [ ]:
# Wrap M_eff back into an xarray object so it carries coordinates again.
m_effective = xr.DataArray(
    m_effective_values,
    dims=n_windows_zhu.dims,
    coords=n_windows_zhu.coords,
    name="M_effective",
)

lwc_uncertainty_final = (lwc_uncertainty_single_measured
                         / np.sqrt(m_effective)).where(mask_retrieval)
lwc_uncertainty_final.name = "LWC_uncertainty"
lwc_uncertainty_final.attrs = {
    "units": "g m-3",
    "long_name": "1-sigma LWC retrieval uncertainty",
    "method": "Zhu et al. (2019) Eq. 5-8, sigma_DWR measured from data",
    "note": "dh = N * gate spacing (window span), NOT the gate spacing",
    "sigma_dwr_db": sigma_dwr_measured_db,
    "n_gates_in_window": n_gates_in_window,
}

print(f"Final per-gate LWC uncertainty:")
print(f"  median {float(lwc_uncertainty_final.median()):.3f} g/m3")
print(f"  p05    {float(lwc_uncertainty_final.quantile(0.05)):.3f} g/m3")
print(f"  p95    {float(lwc_uncertainty_final.quantile(0.95)):.3f} g/m3")


 ---

 ## Step 7 — The key result: how many negatives are SIGNIFICANTLY negative?



 This is the Socuellamos framing. A value is only meaningfully negative if it

 lies more than one sigma below zero, i.e. `LWC + sigma_LWC < 0`.

In [ ]:
def significance_report(lwc_field, uncertainty_field, mask, label):
    """Compare the raw negative fraction with the significant negative fraction."""
    lwc_values = lwc_field.where(mask).values
    sigma_values = uncertainty_field.where(mask).values

    valid = np.isfinite(lwc_values) & np.isfinite(sigma_values)
    lwc_valid = lwc_values[valid]
    sigma_valid = sigma_values[valid]

    fraction_negative = (lwc_valid < 0).mean()
    # "significantly negative" = more than 1 sigma below zero
    fraction_significantly_negative = ((lwc_valid + sigma_valid) < 0).mean()
    # of the negatives, how many could be positive within 1 sigma?
    negatives = lwc_valid < 0
    if negatives.sum() > 0:
        fraction_rescued = ((lwc_valid[negatives] + sigma_valid[negatives]) >= 0).mean()
    else:
        fraction_rescued = np.nan

    print(f"--- {label} ---")
    print(f"  n valid gates                       : {lwc_valid.size}")
    print(f"  fraction LWC < 0                    : {fraction_negative:.1%}")
    print(f"  fraction significantly < 0 (>1 sigma): {fraction_significantly_negative:.1%}")
    print(f"  of the negatives, consistent with 0 : {fraction_rescued:.1%}")
    print(f"  (Socuellamos found >95% consistent with zero)")
    print()
    return fraction_negative, fraction_significantly_negative, fraction_rescued


results_zhu = significance_report(lwc_zhu, lwc_uncertainty_final,
                                  mask_retrieval, "Zhu guarded retrieval")
results_dq = significance_report(lwc_dq, lwc_uncertainty_final,
                                 mask_retrieval, "DQ integral retrieval")


 ---

 ## Step 8 — Plots

In [ ]:
# 8a. The uncertainty field itself
fig, ax = plt.subplots(figsize=(14, 5))
lwc_uncertainty_final.plot(
    x="Time", y="range", ax=ax, cmap="magma_r", robust=True,
    cbar_kwargs={"label": "sigma(LWC) [g m$^{-3}$]"})
ax.set_title("Per-gate LWC uncertainty (Zhu Eq. 5-8)")
plt.show()


In [ ]:
# 8b. LWC with error bars, for three example profiles.
# We pick the profile with the largest MWR LWP, one near the median, and one
# near the minimum, so we see a range of conditions.
lwp_mwr_values = ds_retrieval["LWP"].values

# np.nanargmax returns the INDEX of the largest value, ignoring NaNs.
index_high_lwp = int(np.nanargmax(lwp_mwr_values))
index_median_lwp = int(np.nanargmin(np.abs(lwp_mwr_values - np.nanmedian(lwp_mwr_values))))
index_low_lwp = int(np.nanargmin(lwp_mwr_values))

fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)

for ax, time_index, label in zip(
        axes,
        [index_high_lwp, index_median_lwp, index_low_lwp],
        ["max MWR LWP", "median MWR LWP", "min MWR LWP"]):

    lwc_profile = lwc_zhu.isel(Time=time_index).where(mask_retrieval.isel(Time=time_index))
    sigma_profile = lwc_uncertainty_final.isel(Time=time_index)

    ax.errorbar(
        lwc_profile.values, range_values_m,
        xerr=sigma_profile.values,
        fmt="o-", ms=3, lw=1, capsize=2, elinewidth=0.8,
        label="Zhu retrieval",
    )
    ax.axvline(0, color="k", lw=1.0)
    ax.set_xlabel("LWC [g m$^{-3}$]")
    ax.set_title(f"{label}\nMWR LWP = {lwp_mwr_values[time_index]:.0f} g/m$^2$")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Height [m]")
plt.tight_layout()
plt.show()


In [ ]:
# 8c. Histogram of LWC with the uncertainty band marked
lwc_values_flat = lwc_zhu.where(mask_retrieval).values
lwc_values_flat = lwc_values_flat[np.isfinite(lwc_values_flat)]
median_sigma = float(lwc_uncertainty_final.median())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(lwc_values_flat, bins=120, range=(-3, 3), color="steelblue")
ax.axvline(0, color="k", lw=1.2)
# axvspan shades a vertical band between two x values
ax.axvspan(-median_sigma, median_sigma, color="orange", alpha=0.25,
           label=f"$\\pm 1\\sigma$ = $\\pm${median_sigma:.2f} g/m$^3$")
ax.set_xlabel("LWC [g m$^{-3}$]")
ax.set_ylabel("count")
ax.set_title("LWC distribution with the typical uncertainty band")
ax.legend()
plt.show()

print(f"Fraction of all LWC values falling inside the +/-1 sigma band around zero: "
      f"{np.mean(np.abs(lwc_values_flat) < median_sigma):.1%}")
print("These values are statistically indistinguishable from zero.")


 ---

 ## Step 9 — Testing the range-sidelobe hypothesis



 Hogan (2005) warns that FMCW radars may not reach the precision this

 technique needs, because of range sidelobes. Range sidelobes leak power from

 strong gates into neighbouring weak ones, so the contamination should be

 **worst where the reflectivity gradient is steepest** — at cloud base and

 cloud top.



 If local DWR scatter grows with the local reflectivity gradient, that supports

 the sidelobe explanation. If it is flat, sidelobes are probably not the

 dominant problem.

In [ ]:
ze_W_masked = ds_retrieval["Ze_W"].where(mask_retrieval).values

# Local reflectivity gradient magnitude, gate to gate
reflectivity_gradient = np.abs(ze_W_masked[:, 1:] - ze_W_masked[:, :-1])
# Local DWR scatter, same gate pairs
dwr_local_scatter = np.abs(dwr_masked_values[:, 1:] - dwr_masked_values[:, :-1])

valid_pairs = np.isfinite(reflectivity_gradient) & np.isfinite(dwr_local_scatter)
gradient_valid = reflectivity_gradient[valid_pairs]
scatter_valid = dwr_local_scatter[valid_pairs]

# Bin by reflectivity gradient and take the median DWR scatter in each bin.
gradient_bin_edges = np.percentile(gradient_valid, [0, 20, 40, 60, 80, 100])
print("median |dDWR| as a function of local |dZe|:")
print(f"{'|dZe| bin (dB)':<22}{'n':>8}{'median |dDWR| (dB)':>22}")

bin_centres = []
bin_medians = []
for i in range(len(gradient_bin_edges) - 1):
    low, high = gradient_bin_edges[i], gradient_bin_edges[i + 1]
    in_bin = (gradient_valid >= low) & (gradient_valid < high)
    if in_bin.sum() > 10:
        median_scatter = float(np.median(scatter_valid[in_bin]))
        print(f"{low:6.2f} - {high:6.2f}      {in_bin.sum():>8}{median_scatter:>22.3f}")
        bin_centres.append(0.5 * (low + high))
        bin_medians.append(median_scatter)

correlation_gradient_scatter = float(np.corrcoef(gradient_valid, scatter_valid)[0, 1])
print(f"\ncorrelation(|dZe|, |dDWR|) = {correlation_gradient_scatter:+.3f}")
print("Clearly positive -> consistent with range-sidelobe contamination.")
print("Near zero        -> sidelobes are probably not the dominant error.")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(bin_centres, bin_medians, "o-")
ax.set_xlabel("local reflectivity gradient |dZe| [dB per gate]")
ax.set_ylabel("median |dDWR| [dB per gate]")
ax.set_title("Does DWR scatter grow near strong reflectivity gradients?")
ax.grid(alpha=0.3)
plt.show()


 ---

 ## Step 10 — Save

In [ ]:
ds_uncertainty = xr.Dataset(
    {
        "LWC_uncertainty": lwc_uncertainty_final,
        "M_effective": m_effective,
        "delta_kappa": delta_kappa_field,
    },
    attrs={
        "description": "Per-gate LWC retrieval uncertainty, Zhu et al. (2019) Eq. 5-8",
        "sigma_dwr_measured_db": sigma_dwr_measured_db,
        "sigma_dwr_theoretical_db": sigma_dwr_theoretical_db,
        "ratio_measured_to_theoretical": ratio_measured_to_theoretical,
        "n_gates_in_window": n_gates_in_window,
        "window_span_m": window_span_km * 1000.0,
        "gate_spacing_m": gate_spacing_m,
        "dwell_time_s": dwell_time_s,
        "assumed_prf_hz": prf_W_hz,
        "assumed_spectral_width_m_per_s": spectral_width_W_m_per_s,
        "assumed_snr_db": 10 * np.log10(snr_W_linear),
        "note_dh": "dh in Zhu Eq.5 is the WINDOW SPAN (N*gate spacing), not the "
                   "gate spacing as the paper text states. Verified against Hogan Eq.7.",
        "created_by": "06_error_estimation.ipynb",
    },
)

output_path = Path("../data/processed/lwc_uncertainty.nc")
output_path.parent.mkdir(parents=True, exist_ok=True)
ds_uncertainty.to_netcdf(output_path)

print(f"Saved -> {output_path}")
print(ds_uncertainty)


 ---

 ## Summary for the thesis



 | quantity | value |

 |---|---|

 | measured sigma(DWR) | see Step 2 |

 | theoretical sigma(DWR) (Zhu Eq. 6) | see Step 3 |

 | ratio | see Step 4 |

 | median sigma(LWC) | see Step 6 |

 | fraction LWC < 0 | see Step 7 |

 | fraction significantly < 0 | see Step 7 |



 **Assumptions to state explicitly:**



 1. PRF = 6250 Hz (from Hogan's worked example; an FMCW radar has no true PRF)

 2. Spectral width = 0.3 m/s (Hogan's stratocumulus value)

 3. SNR = 20 dB, or per-gate from `Ze - SL` if the sensitivity limits are used

 4. Range gates are statistically independent

 5. Errors are Gaussian

 6. Zhu Eq. (6) assumes a pulsed radar with no spectral processing; ours is

    FMCW, so the theoretical value is a lower bound rather than an estimate